In [ ]:
! pip install transformers datasets torch evaluate scikit-learn sentencepiece huggingface_hub

In [41]:
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. Load the Dataset (SNLI)
dataset = load_dataset("snli")

# CRITICAL: Re-filtering and removing any rows where label is not 0, 1, or 2
dataset = dataset.filter(lambda x: x['label'] in [0, 1, 2])

# Taking a subset for training/evaluation
train_dataset = dataset["train"].shuffle(seed=42).select(range(30000))
eval_dataset = dataset["validation"].select(range(2000))

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/550152 [00:00<?, ? examples/s]

In [43]:
# 2. Initialize the DeBERTa Tokenizer and Model
model_name = "microsoft/deberta-v3-small"

# DeBERTa v3 requires sentencepiece.
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

# Initialize model with 3 labels for SNLI
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# 3. Define the Tokenization Function
def tokenize_function(examples):
    return tokenizer(
        examples["premise"],
        examples["hypothesis"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Apply tokenization
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight     

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [44]:
# 4. Define Evaluation Metrics
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 5. Define Training Arguments
# Reduced learning rate and added warmup to stabilize DeBERTa training
training_args = TrainingArguments(
    output_dir="./deberta_snli",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=500,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100
)

# 6. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

In [39]:
import torch
print(torch.cuda.is_available()) # Must return True

True


In [45]:
# 7. Execute Training Protocol
trainer.train()

# 8. Final Evaluation
print(trainer.evaluate())

Epoch,Training Loss,Validation Loss,Accuracy
1,0.000000,nan,0.331500
2,0.000000,nan,0.331500
3,0.000000,nan,0.331500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': nan, 'eval_accuracy': 0.3315, 'eval_runtime': 3.2479, 'eval_samples_per_second': 615.791, 'eval_steps_per_second': 38.487, 'epoch': 3.0}


In [30]:
# 7. Execute Training Protocol
trainer.train()

# 8. Final Evaluation
print(trainer.evaluate())

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.177613,0.338000
2,No log,1.100341,0.344000
3,No log,1.102517,0.338000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

{'eval_loss': 1.1002143621444702, 'eval_accuracy': 0.344, 'eval_runtime': 0.9129, 'eval_samples_per_second': 547.69, 'eval_steps_per_second': 35.052, 'epoch': 3.0}


## Hugging Face Inference API check

This optional section exercises a hosted DeBERTa model through the HF Inference API so the notebook can run a remote model call without changing the training cells above.


In [ ]:
import os
from getpass import getpass

from huggingface_hub import InferenceClient

# 9. Hugging Face Inference API demo
# This keeps the notebook self-contained while exercising a hosted DeBERTa model.
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    try:
        HF_TOKEN = getpass("HF_TOKEN (press Enter to skip): " ).strip()
    except Exception:
        HF_TOKEN = ""

if not HF_TOKEN:
    print("HF_TOKEN is not set; skipping the Hugging Face Inference API demo.")
else:
    client = InferenceClient(token=HF_TOKEN)
    model_id = "microsoft/deberta-v3-small"
    prompts = [
        "Dynamic padding is <mask> for faster fine-tuning.",
        "A good SNLI training loop should report <mask> loss every epoch.",
        "The DeBERTa backbone is best used with <mask> tokenization for this demo.",
    ]

    print(f"Querying {model_id} through the HF Inference API")
    for prompt in prompts:
        print(f"\nPrompt: {prompt}")
        try:
            predictions = client.fill_mask(prompt, model=model_id, top_k=3)
            for prediction in predictions:
                print(f"  {prediction['token_str']!r} (score={prediction['score']:.4f})")
        except Exception as exc:
            print(f"  API call failed: {exc}")

Querying Elron/deberta-v3-large-sentiment through the HF Inference API

Prompt: Dynamic padding is <mask> for faster fine-tuning.
  API call failed: 

Prompt: A good SNLI training loop should report <mask> loss every epoch.
  API call failed: 

Prompt: The DeBERTa backbone is best used with <mask> tokenization for this demo.
  API call failed: 
